# Notebook 1 · Data analysis and the four detector families

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
DEBUG = False

os.environ["VISDET_DEBUG"] = "1" if DEBUG else "0"
if os.path.isdir("/kaggle/working"):
    if not os.path.isdir("/kaggle/working/repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "/kaggle/working/repo"], check=True)
    CODE = "/kaggle/working/repo/code"
else:
    CODE = os.environ.get("VISDET_CODE") or os.path.abspath("..")
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
if os.environ.get("VISDET_SKIP_PIP") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

## 01 · Data understanding and preprocessing

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from visdet.common import CLASSES, LONG_SIDE, TRAIN_STEP, VAL_STEP, data_dir, find_comp_root, work_dir
from visdet import eda, prep, viz

COMP = find_comp_root()
FIG = work_dir() / "figures"; FIG.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", font_scale=0.9)
print("data:", COMP)


def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

### 1. Structure and the validation split

In [ ]:
frames = pd.concat([prep.list_frames(COMP, "train"), prep.list_frames(COMP, "val")], ignore_index=True)
test = pd.read_csv(COMP / "test_images.csv")
summary = frames.groupby("split").agg(clips=("clip", "nunique"), frames=("stem", "size"))
summary.loc["test"] = [test["clip"].nunique(), len(test)]
print(summary)
overlap = set(frames.query("split=='train'")["clip"]) & set(frames.query("split=='val'")["clip"])
print("clips shared by train and val:", len(overlap))

In [ ]:
boxes, fstats = eda.load_boxes(frames)
print(f"{len(boxes):,} boxes in {len(fstats):,} frames")
boxes.head()

### 2. Class balance

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
cnt = boxes.groupby(["class", "split"]).size().unstack().reindex(CLASSES)
(cnt / cnt.sum()).plot.bar(ax=ax[0]); ax[0].set_title("Class share per split"); ax[0].set_ylabel("fraction of boxes")
clip_mix = boxes.pivot_table(index="clip", columns="class", values="cls", aggfunc="size", fill_value=0)[CLASSES]
clip_mix = clip_mix.div(clip_mix.sum(axis=1), axis=0)
sns.heatmap(clip_mix.sort_values("car").T, ax=ax[1], cmap="viridis", cbar_kws={"label": "share"}, xticklabels=False)
ax[1].set_title("Class mix per clip (columns = clips)")
save(fig, "eda_classes")
print((cnt.sum(axis=1) / cnt.values.sum()).round(4).to_string())

### 3. Object sizes

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
bins = np.logspace(0, np.log10(boxes["sqrt_area"].max() + 1), 60)
for s, g in boxes.groupby("split"):
    ax[0].hist(g["sqrt_area"], bins=bins, alpha=.6, density=True, label=s)
ax[0].set_xscale("log"); ax[0].axvline(32, c="k", ls="--"); ax[0].axvline(96, c="k", ls=":")
ax[0].set_title("√area (original px); dashed = COCO small/medium"); ax[0].legend()
sns.boxplot(data=boxes, x="class", y="sqrt_area_in", order=CLASSES, ax=ax[1], showfliers=False)
ax[1].set_title("√area at 1280-px input, per class"); ax[1].tick_params(axis="x", rotation=45)
med = boxes.groupby("clip").agg(med=("sqrt_area_in", "median"), split=("split", "first")).sort_values("med")
ax[2].bar(range(len(med)), med["med"], color=np.where(med["split"] == "val", "tab:orange", "tab:blue"))
ax[2].set_title("median √area at input per clip (orange = val)"); ax[2].set_xlabel("clip")
save(fig, "eda_sizes")
sz = boxes.groupby("split")["size_bin"].value_counts(normalize=True).unstack().round(3)
print(sz); print(boxes["sqrt_area_in"].describe(percentiles=[.05, .25, .5, .75, .95]).round(1))

### 4. Scene density: objects per frame

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
sns.histplot(data=fstats, x="n_obj", hue="split", bins=60, stat="density", common_norm=False, ax=ax[0])
ax[0].set_title("Objects per frame")
order = fstats.groupby("clip")["n_obj"].median().sort_values().index
sns.boxplot(data=fstats, x="clip", y="n_obj", order=order, ax=ax[1], showfliers=False,
            hue="split", dodge=False, legend=False)
ax[1].set_xticks([]); ax[1].set_title("Objects per frame by clip (sorted)")
save(fig, "eda_density")
print(fstats.groupby("split")["n_obj"].describe(percentiles=[.5, .9, .99]).round(1))

### 5. Resolutions and ignore regions

In [ ]:
res = fstats.groupby(["split", "W", "H"])["clip"].nunique().rename("clips")
print(res.to_string()); print("test:", test.groupby(["width", "height"])["clip"].nunique().to_dict())
ign = fstats.assign(has=fstats["n_ignore"] > 0).groupby("split").agg(
    frames_with_ignore=("has", "mean"), mean_regions=("n_ignore", "mean"), mean_area_frac=("ignore_frac", "mean"))
print(ign.round(3))

### 6. Temporal redundancy → frame step

In [ ]:
tr_clips = frames.query("split=='train'")["clip"].drop_duplicates()
clips = tr_clips.sample(min(6, len(tr_clips)), random_state=0)
sim = eda.frame_similarity(frames, clips)
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.lineplot(data=sim, x="gap", y="mad", hue="clip", marker="o", ax=ax, legend=False)
ax.set_title("Frame difference vs gap (6 train clips)"); ax.set_ylabel("mean |Δ| grey level")
save(fig, "eda_temporal")
print(sim.groupby("gap")["mad"].mean().round(2).to_string())
print(f"Comparison runs: every {TRAIN_STEP}th train frame -> {len(frames.query('split==\"train\"')) // TRAIN_STEP} frames; "
      f"val: every {VAL_STEP}th frame -> {len(frames.query('split==\"val\"')) // VAL_STEP} frames")

### 7. Anchor coverage

In [ ]:
from visdet.configs import FRCNN_ANCHORS, RETINA_BASE
wh = boxes.query("split=='train'")[["w_in", "h_in"]].to_numpy(np.float32)
cfgs = {
    "FRCNN default (32–512)": eda.anchor_shapes([s[0] for s in FRCNN_ANCHORS["default"]]),
    "FRCNN small (16–256)": eda.anchor_shapes([s[0] for s in FRCNN_ANCHORS["small"]]),
    "RetinaNet default": eda.anchor_shapes([int(x * 2 ** (k / 3)) for x in RETINA_BASE["default"] for k in range(3)]),
    "RetinaNet small": eda.anchor_shapes([int(x * 2 ** (k / 3)) for x in RETINA_BASE["small"] for k in range(3)]),
}
cov = pd.DataFrame({name: {f"IoU>={t}": float((eda.best_anchor_iou(wh, a) >= t).mean()) for t in (0.4, 0.5, 0.6)}
                    for name, a in cfgs.items()}).T
print(cov.round(3))
km = eda.kmeans_iou(wh[np.random.default_rng(0).choice(len(wh), min(len(wh), 50000), replace=False)], k=9)
print("k-means (w,h) at input scale:", np.round(km, 1).tolist())
print("aspect ratio h/w percentiles (5/50/95):", np.percentile(1 / boxes["aspect"], [5, 50, 95]).round(2))
fig, ax = plt.subplots(figsize=(5.5, 4))
for name, a in cfgs.items():
    if "RetinaNet" in name:
        ax.hist(eda.best_anchor_iou(wh, a), bins=50, histtype="step", density=True, label=name, lw=1.5)
ax.axvline(0.5, c="k", ls="--"); ax.set_xlabel("best anchor IoU"); ax.legend(); ax.set_title("RetinaNet anchor coverage")
save(fig, "eda_anchor_coverage")

### 8. Conversion and masking of ignore regions

In [ ]:
demo_dir = work_dir() / "demo_prep"
for sub in ("images/demo", "labels/demo"):
    (demo_dir / sub).mkdir(parents=True, exist_ok=True)
f_ign = fstats[fstats["n_ignore"] > 0].sort_values("n_obj", ascending=False)
pick = frames.set_index("stem").loc[f_ign["stem"].iloc[[0, len(f_ign) // 2]]].reset_index()
fig, ax = plt.subplots(len(pick), 2, figsize=(16, 5 * len(pick)))
ax = np.atleast_2d(ax)
for i, r in enumerate(pick.itertuples()):
    img = viz.rgb(r.src_img); H, W = img.shape[:2]
    lab = prep.read_yolo(r.src_label); reg = prep.cxcywh_to_xyxy(prep.read_ignore(r.src_ignore), W, H)
    a = viz.draw_regions(viz.draw_boxes(img, prep.cxcywh_to_xyxy(lab[:, 1:], W, H), lab[:, 0]), reg, 0.5)
    ax[i, 0].imshow(a); ax[i, 0].set_title(f"{r.stem}: GT boxes + ignore regions (grey)"); viz.legend(ax[i, 0])
    out = prep._export_one({**r._asdict(), "split": "demo"}, demo_dir, LONG_SIDE, True)
    im2 = viz.rgb(out["img"]); h2, w2 = im2.shape[:2]
    lab2 = prep.read_yolo(out["label"])
    ax[i, 1].imshow(viz.draw_boxes(im2, prep.cxcywh_to_xyxy(lab2[:, 1:], w2, h2), lab2[:, 0]))
    ax[i, 1].set_title(f"converted: {w2}x{h2}, masked, {out['n_boxes']} boxes ({out['n_dropped']} dropped)")
for a_ in ax.ravel():
    a_.axis("off")
save(fig, "eda_conversion")

#### Torchvision target check

In [ ]:
from visdet.tv_train import DetDataset
man = pd.DataFrame([dict(img=out["img"], label=out["label"])])
img_t, tgt = DetDataset(man, offset=1)[0]
print({k: v.shape for k, v in tgt.items()}, "labels:", tgt["labels"].unique().tolist())
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(viz.draw_boxes((img_t.permute(1, 2, 0).numpy() * 255).astype(np.uint8), tgt["boxes"].numpy(),
                         tgt["labels"].numpy() - 1)); ax.axis("off"); ax.set_title("torchvision target (xyxy px)")
save(fig, "eda_tv_target")

### 9. Observations → decisions

## Part 2 · Training all detectors

In [ ]:
from visdet.common import TRAIN_STEP, VAL_STEP, data_dir, setup_wandb_from_kaggle_secret
from visdet.configs import BUDGET_H, TV_EXPS, UL_EXPS
from visdet.runner import run_jobs, tv_job, ul_job

setup_wandb_from_kaggle_secret()
DATA, RUNS = data_dir(), work_dir() / "runs"
tr, va, yaml_path = prep.prepare_data(COMP, DATA, TRAIN_STEP, VAL_STEP)
print(f"train {len(tr)} frames from {tr['clip'].nunique()} clips | val {len(va)} frames from {va['clip'].nunique()} clips")
jobs = [tv_job(e, DATA, RUNS) for e in TV_EXPS] + [ul_job(e, yaml_path, RUNS) for e in UL_EXPS]
print(len(jobs), "runs x", BUDGET_H, "GPU-h")
codes = run_jobs(jobs, RUNS / "logs", report_every_s=1800)
print({k: ("OK" if v == 0 else "FAILED") for k, v in codes.items()})

## 02 · Two-stage anchor-based detector: Faster R-CNN (ResNet-50-FPN, torchvision)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from visdet.common import (TRAIN_STEP, VAL_STEP, data_dir, find_comp_root, setup_wandb_from_kaggle_secret,
                           work_dir)
from visdet.configs import BUDGET_H, TV_COMMON, TV_EXPS
from visdet import evaluate as ev, prep, viz
from visdet.results import collect, export_results
from visdet.runner import run_jobs, tv_job

setup_wandb_from_kaggle_secret()
COMP, DATA, RUNS = find_comp_root(), data_dir(), work_dir() / "runs"
FIG = work_dir() / "figures"; FIG.mkdir(exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight"); plt.show()

### Data (same export for every notebook, ~2 min on CPU)

In [ ]:
tr, va, yaml_path = prep.prepare_data(COMP, DATA, TRAIN_STEP, VAL_STEP)
print(f"train frames {len(tr)} from {tr["clip"].nunique()} clips | val frames {len(va)} from {va["clip"].nunique()} clips")
print(f"boxes dropped inside ignore regions: {tr.n_dropped.sum()}")
GT = ev.build_gt(va)

### Training (4 runs, 2 at a time)

In [ ]:
EXPS = [e for e in TV_EXPS if e.startswith("frcnn")]
print("common:", TV_COMMON, "| budget per run:", BUDGET_H, "h")
pd.DataFrame({e: TV_EXPS[e] for e in EXPS}).T.fillna("-")

### Results on the validation clips

In [ ]:
res = collect(["frcnn"], extra=[str(RUNS)])
res.style.format(precision=4).highlight_max(subset=["mAP", "AP50", "AP_small"], color="#cde")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for e in EXPS:
    h = RUNS / e / "history.csv"
    if h.exists():
        h = pd.read_csv(h)
        ax[0].plot(h["train_minutes"], h["mAP"], marker="o", label=e)
        ax[1].plot(h["train_minutes"], h["loss"], marker="o", label=e)
ax[0].set(xlabel="training minutes (one T4)", ylabel="val mAP@[.5:.95]", title="Validation mAP during training")
ax[1].set(xlabel="training minutes", ylabel="mean train loss", title="Training loss"); ax[0].legend(fontsize=8)
save(fig, "frcnn_curves")

In [ ]:
BEST = res.sort_values("mAP", ascending=False)["exp"].iloc[0]
print("best Faster R-CNN run:", BEST)
cls_cols = [c for c in res.columns if c.startswith("AP_") and c not in ("AP_small", "AP_medium", "AP_large")]
res.set_index("exp")[cls_cols].round(3)

### Load the best model

In [ ]:
from visdet.tv_models import build_tv_model, rpn_proposal_recall
from visdet.tv_train import load_rgb, predict_tv
cfg = {**TV_COMMON, **TV_EXPS[BEST]}
model = build_tv_model(cfg).to(DEV)
model.load_state_dict(torch.load(RUNS / BEST / "model_final.pth", map_location=DEV))
model.eval()
preds = {k: {kk: np.asarray(vv) for kk, vv in v.items()} for k, v in json.load(open(RUNS / BEST / "val_preds.json")).items()}

### What each stage contributes

In [ ]:
G = ev.gt_per_image(GT)
sub = va.iloc[::3]
rec = rpn_proposal_recall(model, sub["src_img"].tolist(), [G[s] for s in sub["stem"]], top_ns=(100, 300, 1000), device=DEV)
final = ev.error_breakdown(GT, {s: preds[s] for s in sub["stem"] if s in preds}, score_thr=0.05)[0]["recall_by_size"]
tab = pd.DataFrame({f"RPN top-{n}": r for n, r in rec.items()})
tab["final dets (score≥0.05)"] = pd.Series(final)
tab.round(3)

#### Inference-time number of proposals (no retraining)

In [ ]:
rows = []
for pre, post in [(1000, 300), (1000, 1000), (3000, 2000)]:
    model.rpn._pre_nms_top_n["testing"], model.rpn._post_nms_top_n["testing"] = pre, post
    p = predict_tv(model, va["src_img"].tolist(), va["stem"].tolist(), 1, DEV)
    m = ev.coco_eval(GT, p, per_class=False)
    img0 = load_rgb(va["src_img"].iloc[0]).to(DEV)
    ms = ev.time_inference(lambda x: model([x]), [img0] * 20, warmup=3, n=15)
    rows.append(dict(pre_nms=pre, post_nms=post, mAP=m["mAP"], AP50=m["AP50"], AP_small=m["AP_small"], ms=ms))
model.rpn._pre_nms_top_n["testing"], model.rpn._post_nms_top_n["testing"] = 1000, 1000
pd.DataFrame(rows).round(4)

### Failure analysis (TIDE-style breakdown at score ≥ 0.3)

In [ ]:
summ, per_img = ev.error_breakdown(GT, preds, score_thr=0.3)
print("detections:", summ["counts"]); print("recall by size:", {k: round(v, 3) for k, v in summ["recall_by_size"].items()})
conf = pd.DataFrame(summ["confusion"], index=[f"gt_{c}" for c in viz.CLASSES], columns=viz.CLASSES)
conf.loc[conf.sum(axis=1) > 0, conf.sum(axis=0) > 0]

In [ ]:
cases = {
    "missed": per_img.sort_values("n_missed", ascending=False)["stem"].iloc[0],
    "class confusion": per_img.sort_values("Cls", ascending=False)["stem"].iloc[0],
    "false positives": per_img.assign(fp=per_img["Bkg"] + per_img["Dup"]).sort_values("fp", ascending=False)["stem"].iloc[0],
}
src = va.set_index("stem")["src_img"]
fig, axes = plt.subplots(len(cases), 1, figsize=(16, 9 * len(cases)))
for ax, (name, stem) in zip(np.atleast_1d(axes), cases.items()):
    gb, gl, ig = G[stem]; p = preds[stem]
    img, n_miss = viz.error_overlay(viz.rgb(src[stem]), gb, gl, p["boxes"], p["scores"], p["labels"])
    ax.imshow(viz.draw_regions(img, ig)); ax.axis("off")
    r = per_img.set_index("stem").loc[stem]
    ax.set_title(f"[{name}] {stem}: {int(r.n_gt)} GT, missed {n_miss}, FP {int(r.Bkg)}, dup {int(r.Dup)}, cls-err {int(r.Cls)}")
save(fig, "frcnn_failures")

### Test-set submission with the best Faster R-CNN

In [ ]:
from visdet.infer import check_submission, write_submission
test = pd.read_csv(COMP / "test_images.csv")
tp = predict_tv(model, [str(COMP / f) for f in test["file_name"]], test["image_id"].tolist(), 1, DEV)
SUBS = work_dir() / "submissions"; SUBS.mkdir(exist_ok=True)
sub = write_submission(tp, test, SUBS / f"sub_{BEST}.csv")
check_submission(sub, test)

## 03 · One-stage anchor-based detector: RetinaNet (ResNet-50-FPN, torchvision)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from visdet.common import TRAIN_STEP, VAL_STEP, data_dir, find_comp_root, setup_wandb_from_kaggle_secret, work_dir
from visdet.configs import BUDGET_H, TV_COMMON, TV_EXPS
from visdet import evaluate as ev, prep, viz
from visdet.results import collect, export_results
from visdet.runner import run_jobs, tv_job

setup_wandb_from_kaggle_secret()
COMP, DATA, RUNS = find_comp_root(), data_dir(), work_dir() / "runs"
FIG = work_dir() / "figures"; FIG.mkdir(exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
tr, va, yaml_path = prep.prepare_data(COMP, DATA, TRAIN_STEP, VAL_STEP)
GT = ev.build_gt(va)
G = ev.gt_per_image(GT)
print(f"train {len(tr)} frames | val {len(va)} frames")

### Why the proposal stage matters: anchor statistics

In [ ]:
from visdet.tv_models import retina_anchor_stats
sub = va.iloc[::4]
hw = [(int(r.orig_h), int(r.orig_w)) for r in sub.itertuples()]
gts = [G[s][0] for s in sub["stem"]]
st = {}
for name, a, fg, bg in [("default, 0.5/0.4", "default", .5, .4), ("small, 0.5/0.4", "small", .5, .4),
                        ("small, 0.4/0.3", "small", .4, .3)]:
    d = retina_anchor_stats(a, gts, hw, fg, bg)
    st[name] = dict(anchors_per_img=d["anchors"].mean(), pos_per_img=d["pos"].mean(), neg_per_img=d["neg"].mean(),
                    neg_to_pos=d["neg"].sum() / max(d["pos"].sum(), 1),
                    gt_without_pos_anchor=d["gt_no_anchor"].sum() / d["gt"].sum())
pd.DataFrame(st).T.round(3)

### Training

In [ ]:
EXPS = [e for e in TV_EXPS if e.startswith("retina")]
print("common:", TV_COMMON, "| budget per run:", BUDGET_H, "h")
pd.DataFrame({e: TV_EXPS[e] for e in EXPS}).T.fillna("-")

### Results and comparison with the tuned two-stage detector

In [ ]:
res = collect(["retina", "frcnn"], extra=[str(RUNS)])
res.style.format(precision=4).highlight_max(subset=["mAP", "AP50", "AP_small"], color="#cde")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for e in EXPS:
    h = RUNS / e / "history.csv"
    if h.exists():
        h = pd.read_csv(h)
        ax[0].plot(h["train_minutes"], h["mAP"], marker="o", label=e); ax[1].plot(h["train_minutes"], h["loss"], marker="o", label=e)
ax[0].set(xlabel="training minutes", ylabel="val mAP", title="Validation mAP"); ax[0].legend(fontsize=8)
ax[1].set(xlabel="training minutes", ylabel="train loss", title="Training loss")
save(fig, "retina_curves")

In [ ]:
BEST = res[res.exp.str.startswith("retina")].sort_values("mAP", ascending=False)["exp"].iloc[0]
fr = res[res.exp.str.startswith("frcnn")]
cmp = res[res.exp.isin([e for e in [EXPS[0], BEST] + (fr.sort_values("mAP")["exp"].iloc[-1:].tolist() if len(fr) else [])])]
cmp.set_index("exp")[["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "ms_per_img"]].round(4)

### Error profile of the best RetinaNet vs. the best Faster R-CNN

In [ ]:
from visdet.results import find_run, load_preds
rows = {}
for e in [BEST] + (fr.sort_values("mAP")["exp"].iloc[-1:].tolist() if len(fr) else []):
    run = find_run(e, extra=[str(RUNS)])
    p = load_preds(run) if run else None
    if p:
        s, _ = ev.error_breakdown(GT, p, score_thr=0.3)
        rows[e] = {**s["counts"], **{f"recall_{k}": v for k, v in s["recall_by_size"].items()}}
pd.DataFrame(rows).T.round(3)

### Test-set submission with the best RetinaNet

In [ ]:
from visdet.infer import check_submission, write_submission
from visdet.tv_models import build_tv_model
from visdet.tv_train import predict_tv
model = build_tv_model({**TV_COMMON, **TV_EXPS[BEST]}).to(DEV)
model.load_state_dict(torch.load(RUNS / BEST / "model_final.pth", map_location=DEV)); model.eval()
test = pd.read_csv(COMP / "test_images.csv")
tp = predict_tv(model, [str(COMP / f) for f in test["file_name"]], test["image_id"].tolist(), 0, DEV)
SUBS = work_dir() / "submissions"; SUBS.mkdir(exist_ok=True)
check_submission(write_submission(tp, test, SUBS / f"sub_{BEST}.csv"), test)

## 04 · One-stage anchor-free detector: YOLO11-s (Ultralytics)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from visdet.common import TRAIN_STEP, VAL_STEP, data_dir, find_comp_root, setup_wandb_from_kaggle_secret, work_dir
from visdet.configs import BUDGET_H, UL_COMMON, UL_EXPS
from visdet import evaluate as ev, prep, viz
from visdet.results import collect, export_results, find_run, load_preds
from visdet.runner import run_jobs, ul_job

setup_wandb_from_kaggle_secret()
COMP, DATA, RUNS = find_comp_root(), data_dir(), work_dir() / "runs"
FIG = work_dir() / "figures"; FIG.mkdir(exist_ok=True)


def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
tr, va, yaml_path = prep.prepare_data(COMP, DATA, TRAIN_STEP, VAL_STEP)
GT = ev.build_gt(va)
print(f"train {len(tr)} frames | val {len(va)} frames | {yaml_path}")

### Training

In [ ]:
EXPS = [e for e in UL_EXPS if e.startswith("yolo")]
print("common:", UL_COMMON, "| budget per run:", BUDGET_H, "h")
pd.DataFrame({e: UL_EXPS[e] for e in EXPS}).T.fillna("-")

### Results (by object size) and comparison with the tuned anchor-based detectors

In [ ]:
res = collect(["yolo", "retina", "frcnn"], extra=[str(RUNS)])
res.style.format(precision=4).highlight_max(subset=["mAP", "AP50", "AP_small"], color="#cde")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for e in EXPS:
    f = RUNS / e / "results.csv"
    if f.exists():
        h = pd.read_csv(f); h.columns = [c.strip() for c in h.columns]
        x = h["time"] / 60 if "time" in h else h["epoch"]
        ax[0].plot(x, h["metrics/mAP50-95(B)"], marker="o", label=e)
        ax[1].plot(x, h["train/box_loss"], marker="o", label=e)
ax[0].set(xlabel="training minutes", ylabel="mAP50-95 (Ultralytics val, no ignore regions)", title="Validation during training")
ax[1].set(xlabel="training minutes", ylabel="box loss", title="Training box loss"); ax[0].legend(fontsize=8)
save(fig, "yolo_curves")

In [ ]:
BEST = res[res.exp.str.startswith("yolo")].sort_values("mAP", ascending=False)["exp"].iloc[0]
best_of = lambda pre: res[res.exp.str.startswith(pre)].sort_values("mAP")["exp"].iloc[-1:].tolist()
sel = ["yolo_E0_baseline", BEST] + best_of("retina") + best_of("frcnn")
tab = res.set_index("exp").loc[[e for e in dict.fromkeys(sel) if e in set(res.exp)]]
tab[["mAP", "AP50", "AP_small", "AP_medium", "AP_large", "ms_per_img"]].round(4)

In [ ]:
rows = {}
for e in tab.index:
    run = find_run(e, extra=[str(RUNS)])
    p = load_preds(run) if run else None
    if p:
        s, _ = ev.error_breakdown(GT, p, score_thr=0.3)
        rows[e] = {**{f"recall_{k}": v for k, v in s["recall_by_size"].items()}, **s["counts"]}
size_tab = pd.DataFrame(rows).T
size_tab.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
tab[["AP_small", "AP_medium", "AP_large"]].plot.bar(ax=ax); ax.set_ylabel("AP"); ax.set_title("AP by object size")
ax.tick_params(axis="x", rotation=20)
save(fig, "yolo_vs_anchor_by_size")

### Input resolution at inference (best model, no retraining)

In [ ]:
from visdet.ul_train import load_trained, predict_ul
model = load_trained(RUNS / BEST / "best.pt")
rows = []
for sz in [1024, 1280, 1536]:
    p = predict_ul(model, va["src_img"].tolist(), va["stem"].tolist(), sz)
    m = ev.coco_eval(GT, p, per_class=False)
    rows.append(dict(imgsz=sz, mAP=m["mAP"], AP50=m["AP50"], AP_small=m["AP_small"], AP_medium=m["AP_medium"]))
pd.DataFrame(rows).round(4)

### Dense scenes: recall vs. number of objects in the frame

In [ ]:
p_best = load_preds(RUNS / BEST)
_, per_img = ev.error_breakdown(GT, p_best, score_thr=0.3)
per_img["density"] = pd.cut(per_img["n_gt"], [0, 25, 50, 100, 200, 1000])
d = per_img.groupby("density", observed=True).apply(lambda g: 1 - g.n_missed.sum() / max(g.n_gt.sum(), 1)).rename("recall")
d.to_frame().assign(frames=per_img.groupby("density", observed=True).size()).round(3)

### Test-set submission with the best YOLO run

In [ ]:
from visdet.infer import check_submission, write_submission
test = pd.read_csv(COMP / "test_images.csv")
tp = predict_ul(model, [str(COMP / f) for f in test["file_name"]], test["image_id"].tolist(), UL_COMMON["imgsz"])
SUBS = work_dir() / "submissions"; SUBS.mkdir(exist_ok=True)
check_submission(write_submission(tp, test, SUBS / f"sub_{BEST}.csv"), test)

## 05 · Query-based detector: RT-DETR-L (Ultralytics)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from visdet.common import TRAIN_STEP, VAL_STEP, data_dir, find_comp_root, setup_wandb_from_kaggle_secret, work_dir
from visdet.configs import BUDGET_H, UL_COMMON, UL_EXPS
from visdet import evaluate as ev, prep
from visdet.results import collect, export_results, find_run, load_preds
from visdet.runner import run_jobs, ul_job

setup_wandb_from_kaggle_secret()
COMP, DATA, RUNS = find_comp_root(), data_dir(), work_dir() / "runs"
FIG = work_dir() / "figures"; FIG.mkdir(exist_ok=True)


def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
tr, va, yaml_path = prep.prepare_data(COMP, DATA, TRAIN_STEP, VAL_STEP)
GT = ev.build_gt(va)
print(f"train {len(tr)} frames | val {len(va)} frames")

In [ ]:
EXPS = [e for e in UL_EXPS if e.startswith("rtdetr")]
print("common:", UL_COMMON, "| budget per run:", BUDGET_H, "h")
pd.DataFrame({e: UL_EXPS[e] for e in EXPS}).T.fillna("-")

### Results

In [ ]:
res = collect(["rtdetr", "yolo", "retina", "frcnn"], extra=[str(RUNS)])
res.style.format(precision=4).highlight_max(subset=["mAP", "AP50", "AP_small"], color="#cde")

### Validation mAP over training time: RT-DETR vs. YOLO11 under the same budget

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
curves = {e: find_run(e, extra=[str(RUNS)]) for e in EXPS + ["yolo_E0_baseline"]}
for e, run in curves.items():
    if run is not None and (run / "results.csv").exists():
        h = pd.read_csv(run / "results.csv"); h.columns = [c.strip() for c in h.columns]
        x = h["time"] / 60 if "time" in h else h["epoch"]
        ls = "--" if e.startswith("yolo") else "-"
        ax[0].plot(x, h["metrics/mAP50-95(B)"], ls, marker="o", ms=3, label=e)
        if "train/giou_loss" in h:
            ax[1].plot(x, h["train/giou_loss"], ls, label=e)
ax[0].set(xlabel="training minutes (one T4)", ylabel="val mAP50-95", title="Same GPU budget")
ax[1].set(xlabel="training minutes", ylabel="GIoU loss (train)", title="RT-DETR training loss"); ax[0].legend(fontsize=8)
save(fig, "rtdetr_vs_yolo_curve")

### Small objects and dense scenes

In [ ]:
BEST = res[res.exp.str.startswith("rtdetr")].sort_values("mAP", ascending=False)["exp"].iloc[0]
best_yolo = res[res.exp.str.startswith("yolo")].sort_values("mAP")["exp"].iloc[-1:].tolist()
rows, dens = {}, {}
for e in [BEST, "yolo_E0_baseline"] + best_yolo:
    run = find_run(e, extra=[str(RUNS)])
    p = load_preds(run) if run else None
    if not p:
        continue
    s, per = ev.error_breakdown(GT, p, score_thr=0.3)
    rows[e] = {**{f"recall_{k}": v for k, v in s["recall_by_size"].items()}, **s["counts"]}
    per["density"] = pd.cut(per["n_gt"], [0, 25, 50, 100, 200, 1000])
    dens[e] = per.groupby("density", observed=True).apply(lambda g: 1 - g.n_missed.sum() / max(g.n_gt.sum(), 1))
display(pd.DataFrame(rows).T.round(3))
fig, ax = plt.subplots(figsize=(7, 3.5))
pd.DataFrame(dens).plot.bar(ax=ax); ax.set(ylabel="recall @ score≥0.3", xlabel="GT objects in frame", title="Recall vs density")
ax.tick_params(axis="x", rotation=0)
save(fig, "rtdetr_density")

### Test-set submission with the best RT-DETR run

In [ ]:
from visdet.infer import check_submission, write_submission
from visdet.ul_train import load_trained, predict_ul
model = load_trained(RUNS / BEST / "best.pt")
test = pd.read_csv(COMP / "test_images.csv")
tp = predict_ul(model, [str(COMP / f) for f in test["file_name"]], test["image_id"].tolist(), UL_COMMON["imgsz"])
SUBS = work_dir() / "submissions"; SUBS.mkdir(exist_ok=True)
check_submission(write_submission(tp, test, SUBS / f"sub_{BEST}.csv"), test)

In [ ]:
from visdet.results import export_results
export_results(work_dir() / "runs", work_dir() / "results_nb1.zip", figures_dir=FIG)